# 03 — Explore Segmentation (ChickenDet masks + COCO→YOLO conversion)

**Phase 1 (Exploration & Learning)** notebook feeding into **Phase 3 (Segmentation Baseline)** — run on **Google Colab** (free T4 GPU) or the repo's local `.venv`, whichever's free.

**Goal for this notebook:** before training any segmentation model, verify the mask data is actually good and that the COCO→YOLO-seg conversion this project already uses is faithful. Specifically:
1. Load ChickenDet's COCO segmentation masks and check them for basic quality issues (missing/degenerate polygons, mask/box area ratio).
2. Visualize original COCO masks overlaid on images.
3. Run `ultralytics.data.converter.convert_coco(..., use_segments=True)` — the same conversion `src/poultry_monitoring/data/coco.py`'s `convert_coco_to_yolo_labels` wraps in production (already called on every *detection* run too, since `copy_paste` needs segments) — and compare the *converted* YOLO-format polygons against the originals, numerically (mask IoU) and visually.
4. Prototype and visualize candidate augmentations directly against Albumentations (flips, zoom, copy-paste, CLAHE, autocontrast) on random images, tracking mask effects alongside pixel effects — a sandbox for trying ideas, not a visualizer for already-decided `src/` code.

**Out of scope for this notebook**: actually training a `yolo26n-seg` model — that's the next step (`plan.md`'s "prototype segmentation heads" item), deliberately kept separate so this one stays fast/GPU-optional and focused on the data itself.

## Setup

Runs on **Colab** (free T4) or the repo's local `uv`-managed `.venv` — the cell below detects which and adapts installs/paths accordingly.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'local (.venv)'}")

if IN_COLAB:
    get_ipython().system("pip install ultralytics pycocotools")
# Locally, deps come from `uv sync --extra dev` (CLAUDE.md § Commands) — nothing to install here.

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

## Point at ChickenDet

On Colab, reuses the same Drive-mounted copy as [`01_explore_chickenverse.ipynb`](01_explore_chickenverse.ipynb)/[`02_yolo26_baseline.ipynb`](02_yolo26_baseline.ipynb). Locally, uses the repo's own `data/`

In [ ]:
if IN_COLAB:
    DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/poultry_monitoring/data/ChickenDet")
else:
    DATA_DIR = Path.cwd().parent / "data" / "ChickenDet"  # repo-local, run from notebooks/

assert DATA_DIR.exists(), f"Dataset not found at {DATA_DIR}"
print(f"Data directory: {DATA_DIR}")

## Load libraries

In [ ]:
import shutil

import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from pycocotools.coco import COCO
import pycocotools.mask as mask_utils

from ultralytics.data.converter import convert_coco

%matplotlib inline
np.random.seed(42)  # reproducible sample picks below

## Load COCO annotations (masks)

Same annotation files as [`01_explore_chickenverse.ipynb`](01_explore_chickenverse.ipynb) — this notebook picks up specifically on the segmentation side, past that notebook's own boxes-plus-first-look-at-masks scope.

In [ ]:
coco_train = COCO(f"{DATA_DIR}/annotations/instances_Train.json")
coco_val = COCO(f"{DATA_DIR}/annotations/instances_Validation.json")
coco_test = COCO(f"{DATA_DIR}/annotations/instances_Test.json")

SPLITS = {"Train": coco_train, "Validation": coco_val, "Test": coco_test}

## Mask quality checks

Before trusting these masks for training, check for the failure modes that'd actually bite: annotations missing a segmentation entirely (box-only, silently no-ops a `-seg` model's mask loss), degenerate polygons (too few points / zero area), and the ratio of mask area to box area (should be ≤1 and not wildly small — a mask that's a tiny fraction of its own box usually means a labeling problem, not a genuinely sparse bird).

In [ ]:
def check_mask_quality(coco: COCO) -> dict:
    """Scan every annotation for missing/degenerate segmentations and box/mask area ratio."""
    ann_ids = coco.getAnnIds()
    missing_seg = 0
    degenerate = 0
    ratios = []
    vertex_counts = []
    format_counts = {"polygon": 0, "rle": 0}

    for ann in coco.loadAnns(ann_ids):
        seg = ann.get("segmentation")

        if seg is None or (isinstance(seg, list) and len(seg) == 0):
            missing_seg += 1
            continue

        mask_area = 0.0

        if isinstance(seg, list):
            format_counts["polygon"] += 1
            poly = seg[0]
            v_count = len(poly) // 2
            vertex_counts.append(v_count)
            if v_count < 3:
                degenerate += 1
                continue
            rle = mask_utils.frPyObjects(seg, ann['bbox'][3], ann['bbox'][2])
            mask_area = float(mask_utils.area(rle).sum())

        elif isinstance(seg, dict):
            format_counts["rle"] += 1
            rle = mask_utils.frPyObjects([seg], seg['size'][0], seg['size'][1])
            mask_area = float(mask_utils.area(rle)[0])

        else:
            missing_seg += 1
            continue

        if mask_area <= 0:
            degenerate += 1
            continue

        box_area = ann["bbox"][2] * ann["bbox"][3]
        if box_area > 0:
            ratios.append(mask_area / box_area)

    return {
        "n_annotations": len(ann_ids),
        "missing_segmentation": missing_seg,
        "degenerate_polygons": degenerate,
        "vertex_counts": np.array(vertex_counts),
        "mask_to_box_area_ratio": np.array(ratios),
        "formats": format_counts
    }


for name, coco in SPLITS.items():
    stats = check_mask_quality(coco)
    r = stats["mask_to_box_area_ratio"]
    vc = stats["vertex_counts"]
    f = stats["formats"]

    median_vc = f"{np.median(vc):.0f}" if vc.size > 0 else "N/A"
    median_r = f"{np.median(r):.2f}" if r.size > 0 else "N/A"

    print(
        f"{name}: {stats['n_annotations']} total anns | "
        f"Formats: {f['polygon']} poly, {f['rle']} RLE | "
        f"Missing: {stats['missing_segmentation']} | "
        f"Degenerate: {stats['degenerate_polygons']} | "
        f"Vertices (poly only) median={median_vc} | "
        f"Mask/Box ratio median={median_r}"
    )

## Visualize original COCO masks

Overlay masks on a few sample images per split — eyeball whether the polygons actually trace the birds, not just that the counts check out numerically above.

In [ ]:
def visualize_masks(coco: COCO, img_id: int, img_dir: Path, title: str = "") -> None:
    """Display one image with its COCO segmentation masks overlaid."""
    img_info = coco.loadImgs(img_id)[0]
    image = np.array(Image.open(Path(img_dir) / img_info["file_name"]).convert("RGB"))
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(image)
    ax.set_title(f"{title} (image {img_id}, {len(anns)} instances)")
    ax.axis("off")
    coco.showAnns(anns, draw_bbox=False)
    plt.show()


for name, coco in SPLITS.items():
    split_dir = "Validation" if name == "Validation" else name
    img_ids = coco.getImgIds()
    sample_id = img_ids[np.random.randint(0, len(img_ids))]
    visualize_masks(coco, sample_id, DATA_DIR / "images" / split_dir, title=name)

## Run the COCO → YOLO-seg conversion

Same call `src/poultry_monitoring/data/coco.py`'s `convert_coco_to_yolo_labels(..., use_segments=True)` wraps in production — run directly here (into a scratch dir, not the repo's real `data/`) so the raw output is easy to inspect.

### Preprocessing RLE to Polygons

Since the default converter requires polygons, we iterate through the annotation files, decode the RLE masks, extract contours with OpenCV, and replace the RLE dict with a list of polygons.

The result is cached to disk under `DATA_DIR/annotations_polygon_cache` — this is the expensive step (~1-2 min for Train's 116k annotations) and there's no reason to redo it on every notebook run, so a cached split is reused as-is if present (set `FORCE_REBUILD_POLYGON_CACHE = True` below to force a rebuild, e.g. after changing the simplification logic). Any annotation where OpenCV can't extract a usable contour is logged explicitly and left with its original RLE segmentation — `convert_coco` will still silently bbox-fill those, so surfacing them here turns that into a known, counted quantity instead of a silent gap.

In [ ]:
import json

FORCE_REBUILD_POLYGON_CACHE = False  # flip to True after changing the contour/simplification logic below

POLYGON_ANN_DIR = DATA_DIR / "annotations_polygon_cache"  # persistent cache, not scratch -- reused across notebook runs
POLYGON_ANN_DIR.mkdir(parents=True, exist_ok=True)


def convert_rle_to_polygons_in_json(json_path: Path, output_path: Path) -> None:
    """Load a COCO JSON, replace RLE segmentations with polygon lists, and save.

    Annotations where OpenCV can't extract a usable contour (e.g. a fully degenerate
    mask) are left with their original RLE segmentation and reported by id -- `convert_coco`
    silently bbox-fills anything it can't read as a polygon, so this is the only point
    where such instances are visible at all.
    """
    with open(json_path) as f:
        data = json.load(f)

    print(f"Preprocessing {json_path.name}...")
    converted_count = 0
    unconverted = []

    for ann in data["annotations"]:
        seg = ann.get("segmentation")
        if not (isinstance(seg, dict) and "counts" in seg):
            continue  # already a polygon (or missing) -- nothing to do

        rle = mask_utils.frPyObjects([seg], seg["size"][0], seg["size"][1])
        m = mask_utils.decode(rle)[:, :, 0]

        contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        polygons = []
        for cnt in contours:
            # Simplify to cut label-file size while keeping shape detail (~0.5px tolerance)
            poly = cv2.approxPolyDP(cnt, 0.5, True).flatten().tolist()
            if len(poly) >= 6:  # x, y * 3 minimum for a valid polygon
                polygons.append(poly)

        if polygons:
            ann["segmentation"] = polygons
            converted_count += 1
        else:
            unconverted.append((ann["id"], ann["image_id"]))

    with open(output_path, "w") as f:
        json.dump(data, f)

    print(f"  Converted {converted_count} RLE masks to polygons.")
    if unconverted:
        print(
            f"  WARNING: {len(unconverted)} annotation(s) had no usable contour and "
            f"kept their original RLE segmentation -- convert_coco will bbox-fill "
            f"these silently. ann_id(image_id): "
            f"{', '.join(f'{a}({i})' for a, i in unconverted[:10])}"
            f"{' ...' if len(unconverted) > 10 else ''}"
        )


for split in ["Train", "Validation", "Test"]:
    src = DATA_DIR / "annotations" / f"instances_{split}.json"
    dst = POLYGON_ANN_DIR / f"instances_{split}.json"
    if dst.exists() and not FORCE_REBUILD_POLYGON_CACHE:
        print(f"{dst.name}: using cached polygon annotations ({dst})")
        continue
    convert_rle_to_polygons_in_json(src, dst)

### Run YOLO Conversion on Polygon JSONs

Run `convert_coco` against the cached polygon JSONs. This step itself is cheap to redo (plain COCO polygon parsing, no RLE decode) — no need to cache its output separately, just re-run it whenever the polygon cache changes.

In [ ]:
YOLO_LABELS_DIR = DATA_DIR / "notebook_03_scratch_high_fidelity"  # scratch: NOT nested under SCRATCH_DIR below, so its cleanup can't clobber this
if YOLO_LABELS_DIR.exists():
    shutil.rmtree(YOLO_LABELS_DIR)

convert_coco(
    labels_dir=str(POLYGON_ANN_DIR),
    save_dir=str(YOLO_LABELS_DIR),
    use_segments=True,
    use_keypoints=False,
    cls91to80=False,
)

print(f"High-fidelity YOLO labels saved to: {YOLO_LABELS_DIR}")

In [ ]:
SCRATCH_DIR = DATA_DIR / "notebook_03_scratch_low_fidelity"  # scratch: naive conversion straight from the original RLE annotations, kept only to demonstrate the bbox-fallback bug below
if SCRATCH_DIR.exists():
    shutil.rmtree(SCRATCH_DIR)

convert_coco(
    labels_dir=str(DATA_DIR / "annotations"),
    save_dir=str(SCRATCH_DIR),
    use_segments=True,
    use_keypoints=False,
    cls91to80=False,
)

converted_train_labels = SCRATCH_DIR / "labels" / "Train"
print(f"Converted labels written to: {converted_train_labels}")
sample_files = sorted(p.name for p in converted_train_labels.glob("*.txt"))[:5]
print(f"Sample files: {sample_files}")

## Compare converted YOLO polygons against the original COCO masks

Parse one converted label file's normalized polygons back into pixel space, rasterize both the original COCO mask and the converted YOLO polygon for the same instance, and check they actually agree — not just that a `.txt` file got written. `compute_conversion_fidelity` below takes a `label_dir` argument so the same check runs against both the low-fidelity (`SCRATCH_DIR`) and high-fidelity (`YOLO_LABELS_DIR`) conversions further down, instead of duplicating the logic.

In [ ]:
def rasterize_yolo_polygon(line: str, img_w: int, img_h: int) -> np.ndarray:
    """Turn one YOLO-seg label line (`class x1 y1 x2 y2 ...`, normalized) into a binary mask."""
    parts = list(map(float, line.split()))
    if len(parts) < 3:
        return np.zeros((img_h, img_w), dtype=np.uint8)
    coords = np.array(parts[1:]).reshape(-1, 2)
    coords[:, 0] *= img_w
    coords[:, 1] *= img_h
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillPoly(mask, [coords.astype(np.int32)], 1)
    return mask


def decode_rle_mask(coco: COCO, ann: dict) -> np.ndarray:
    """Correctly decode a COCO segmentation (polygon or compressed RLE) into a binary mask."""
    seg = ann["segmentation"]
    if isinstance(seg, list):
        return coco.annToMask(ann)
    rle = mask_utils.frPyObjects([seg], seg["size"][0], seg["size"][1])
    return mask_utils.decode(rle)[:, :, 0]


def mask_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Intersection-over-union of two binary masks."""
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return intersection / union if union > 0 else 0.0


def compute_conversion_fidelity(
    coco: COCO, img_id: int, label_dir: Path, split: str = "Train"
) -> tuple[np.ndarray, list[str]]:
    """Compare one image's original COCO masks against its converted YOLO-seg labels.

    Rasterizes the original segmentation (polygon or RLE) and the converted YOLO-seg
    polygon for each instance and scores them by mask IoU -- the numeric counterpart to
    the visual check in `plot_conversion_comparison`.

    Args:
        coco: COCO object holding the original (pre-conversion) annotations.
        img_id: Image id to check.
        label_dir: Root of a `convert_coco` output (e.g. `SCRATCH_DIR`/`YOLO_LABELS_DIR`).
        split: Split subfolder under `label_dir/labels/`, matching the COCO json used to
            build `coco`.

    Returns:
        Tuple of (per-instance IoU array, converted label lines) -- the lines are
        returned too so callers like `plot_conversion_comparison` don't re-read the file.

    Raises:
        FileNotFoundError: If no converted label file exists for this image.
    """
    img_info = coco.loadImgs(img_id)[0]
    img_w, img_h = img_info["width"], img_info["height"]
    stem = Path(img_info["file_name"]).stem
    label_path = label_dir / "labels" / split / f"{stem}.txt"
    if not label_path.exists():
        raise FileNotFoundError(f"No converted label file at {label_path}")

    lines = label_path.read_text().splitlines()
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
    ious = np.array(
        [
            mask_iou(decode_rle_mask(coco, ann), rasterize_yolo_polygon(line, img_w, img_h))
            for ann, line in zip(anns, lines)
        ]
    )
    return ious, lines


In [ ]:
img_ids = coco_train.getImgIds()
sample_img_id = int(img_ids[np.random.randint(0, len(img_ids))])

In [ ]:
ious, lines = compute_conversion_fidelity(coco_train, sample_img_id, SCRATCH_DIR)
low_fidelity_ious = ious  # preserved for the Summary chart -- SCRATCH_DIR gets deleted by the cleanup cell below, this array doesn't
print(f"Image {sample_img_id}: {len(ious)} instances")
print(f"Fidelity Analysis: IoU mean={ious.mean():.4f}, min={ious.min():.4f}")
if ious.mean() < 0.9:
    print("CONFIRMED: The converter is simplifying RLE masks into BBox-aligned rectangles.")

In [ ]:
# Verification: same check, against the high-fidelity (polygon-cache-based) conversion
ious, lines = compute_conversion_fidelity(coco_train, sample_img_id, YOLO_LABELS_DIR)
high_fidelity_ious = ious  # preserved for the Summary chart
print(f"High-Fidelity Fidelity Analysis: IoU mean={ious.mean():.4f}, min={ious.min():.4f}")
if ious.mean() > 0.95:
    print("VERIFIED: The in-memory polygon conversion preserved mask detail!")

## Side-by-side: original vs. converted, same image

The IoU numbers above should already be ~1.0 (both encode the same polygon, just normalized vs. pixel-space) — this is the visual confirmation.

In [ ]:
def plot_conversion_comparison(
    coco: COCO, img_id: int, label_dir: Path, img_dir: Path, split: str = "Train", title: str = ""
) -> None:
    """Side-by-side: original COCO masks (left) vs. converted YOLO-seg polygons (right),
    one random color per instance on the converted side so instance separation is easy
    to eyeball, not just aggregate coverage.

    Args:
        coco: COCO object holding the original (pre-conversion) annotations.
        img_id: Image id to visualize.
        label_dir: Root of a `convert_coco` output, as in `compute_conversion_fidelity`.
        img_dir: Directory the raw image lives in (e.g. `DATA_DIR / "images" / split`).
        split: Split subfolder under `label_dir/labels/`.
        title: Figure title prefix.
    """
    img_info = coco.loadImgs(img_id)[0]
    img_w, img_h = img_info["width"], img_info["height"]
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
    _, lines = compute_conversion_fidelity(coco, img_id, label_dir, split=split)

    image = np.array(Image.open(Path(img_dir) / img_info["file_name"]).convert("RGB"))

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(image)
    axes[0].set_title("Original COCO masks")
    axes[0].axis("off")
    plt.sca(axes[0])
    coco.showAnns(anns, draw_bbox=False)

    converted_overlay = image.copy()
    rng = np.random.default_rng(0)  # reproducible per-instance colors
    for line in lines:
        mask = rasterize_yolo_polygon(line, img_w, img_h).astype(bool)
        color = rng.integers(50, 255, size=3)
        converted_overlay[mask] = (converted_overlay[mask] * 0.5 + color * 0.5).astype(np.uint8)
    axes[1].imshow(converted_overlay)
    axes[1].set_title("Converted YOLO-seg polygons")
    axes[1].axis("off")

    fig.suptitle(f"{title} (image {img_id}, {len(anns)} instances)")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_conversion_comparison(
    coco_train,
    sample_img_id,
    YOLO_LABELS_DIR,
    DATA_DIR / "images" / "Train",
    title="High-fidelity conversion",
)

In [ ]:
# Scratch conversion output only -- the polygon annotation cache (POLYGON_ANN_DIR) is
# deliberately NOT removed here, so re-running this notebook later reuses it instead of
# repaying the RLE-decode cost.
shutil.rmtree(SCRATCH_DIR, ignore_errors=True)
shutil.rmtree(YOLO_LABELS_DIR, ignore_errors=True)

## Augmentation examples (image + mask effects)

**Deliberately self-contained — no `poultry_monitoring` imports.** This is meant to be a starting point for trying out and prototyping new augmentation ideas directly against Albumentations, not a visualizer for already-decided production code. Anything that proves useful here is a candidate to port into `src/` afterwards, per this project's own notebook-first workflow (`CLAUDE.md` § Source of Truth) — not the other way around.

Covers both **target-aware** transforms (flips, zoom, copy-paste — these move or recompose the masks, not just the pixels) and **image-only** ones (CLAHE, autocontrast). Masks are carried through Albumentations' own `masks`/`mask` targets rather than reusing the original COCO annotations unchanged, so this works correctly for spatial transforms too, not just color/contrast ones.

In [ ]:
import albumentations as A

### Helpers

`get_instance_masks` rasterizes every instance's COCO polygon into its own binary mask (same approach as the conversion-fidelity check above, via `coco.annToMask`). `overlay_masks` draws a list of binary masks as distinct semi-transparent colors, working directly on mask arrays rather than COCO annotation objects — needed once a transform has actually moved the masks, since `coco.showAnns` only knows how to draw the *original*, untransformed annotations. `format_transform_params` reads a transform's non-default constructor args straight off the instance (Albumentations' own serialization introspection) so plot titles always reflect the actual params a transform was built with, without a hand-maintained title string to keep in sync.

In [ ]:
def get_instance_masks(coco: COCO, img_id: int) -> list[np.ndarray]:
    """Rasterize every instance's COCO polygon into its own binary mask."""
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
    return [coco.annToMask(ann) for ann in anns]


def overlay_masks(image: np.ndarray, masks: list[np.ndarray], alpha: float = 0.45) -> np.ndarray:
    """Draw each instance mask as a distinct semi-transparent color, for display only."""
    overlay = image.copy()
    rng = np.random.default_rng(0)
    for mask in masks:
        if mask.sum() == 0:  # e.g. an instance a spatial transform pushed off-frame
            continue
        color = rng.integers(50, 255, size=3)
        region = mask.astype(bool)
        overlay[region] = (overlay[region] * (1 - alpha) + color * alpha).astype(np.uint8)
    return overlay


def format_transform_params(transform: A.BasicTransform) -> str:
    """Render a transform's non-default __init__ params as `key=value, ...`.

    Diffs the transform's own args (via Albumentations' `get_transform_init_args`,
    the same introspection it uses for serialization) against a freshly constructed
    default instance of the same class, so the result only surfaces what was actually
    customized -- e.g. `CLAHE(clip_limit=12.0)` shows `clip_limit=12.0`, not every
    other CLAHE constructor arg too.
    """
    args = transform.get_transform_init_args()
    try:
        defaults = type(transform)().get_transform_init_args()
    except Exception:
        defaults = {}
    skip = {"p", "always_apply"}
    changed = {k: v for k, v in args.items() if k not in skip and v != defaults.get(k, object())}
    return ", ".join(f"{k}={v}" for k, v in changed.items())


def plot_augmentation_comparison(
    image: np.ndarray,
    masks: list[np.ndarray],
    transform: A.BasicTransform,
    title: str = "",
    show_labels: bool = True,
    seed: int | None = None,
) -> None:
    """Show original vs. augmented image, optionally with mask overlays on each, in a
    2x2 grid (or a single row of 2 without labels).

    Runs `transform` through Albumentations' `masks` target, so spatial transforms
    (flips, zoom, ...) move the masks along with the image -- not just non-spatial
    color/contrast ones. Top row is plain original/augmented, bottom row is the same
    pair with mask overlays -- sized to fit on one screen without scrolling. The
    transform's own non-default params (clip_limit, cutoff, brightness_limit, ...) are
    read off the instance and shown in the title, so trying a new value is just
    editing the transform's kwargs in `TRANSFORMS_TO_EXPLORE` below -- no title string
    to keep in sync by hand.

    Args:
        image: RGB image array.
        masks: One binary mask per instance, e.g. from `get_instance_masks`.
        transform: A single Albumentations transform (wrapped in `A.Compose` here).
        title: Prefix for the figure title (typically the transform's dict key).
        show_labels: If True, show all 4 panels (original/augmented x without/with
            masks) in a 2x2 grid. If False, show just 2 (original vs. augmented, no
            masks) -- useful on high-density images where mask overlays clutter the
            comparison.
        seed: Optional seed for a reproducible augmented draw.
    """
    pipeline = A.Compose([transform])
    if seed is not None:
        pipeline.set_random_seed(seed)
    result = pipeline(image=image, masks=masks)
    aug_image, aug_masks = result["image"], result["masks"]

    if show_labels:
        fig, axes = plt.subplots(2, 2, figsize=(12, 9))
        axes = axes.flatten()
        panels = [
            (image, None, "original"),
            (aug_image, None, "augmented"),
            (image, masks, "original + masks"),
            (aug_image, aug_masks, "augmented + masks"),
        ]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        panels = [(image, None, "original"), (aug_image, None, "augmented")]

    for ax, (img, panel_masks, panel_title) in zip(axes, panels):
        display_img = overlay_masks(img, panel_masks) if panel_masks is not None else img
        ax.imshow(display_img)
        ax.axis("off")
        ax.set_title(panel_title)

    params = format_transform_params(transform)
    header = f"{title} ({params})" if params else title
    fig.suptitle(f"{header} -- {len(masks)} instances")
    plt.tight_layout(pad=0.5, w_pad=0.3, h_pad=0.5)
    plt.show()

### Flips, zoom, CLAHE, autocontrast, brightness/contrast, hue-saturation-value

All Albumentations-native, all forced to `p=1.0` so the effect is guaranteed visible on a single draw. `VerticalFlip` is included deliberately, not just `HorizontalFlip` — ChickenVerse is shot directly overhead with no canonical "up", so it's a legitimate axis of variation here, unlike for a typical ground-level photo. `clahe`/`autocontrast`/`brightness_contrast`/`hue_saturation_value` are Albumentations' own built-in versions — deliberately *not* this project's existing custom ones in `augmentation/shared.py`/`preprocessing_eval.py`, since the point here is trying things independently, not re-confirming what's already built.

Each entry's non-default constructor params show up automatically in its plot title (via `format_transform_params` above) — comparing a different `clip_limit`/`cutoff`/etc. is just adding another dict entry with a different value, as `clahe_default`/`clahe_strong` and `autocontrast_default`/`autocontrast_aggressive` do below.

In [ ]:
img_ids = coco_train.getImgIds()
sample_img_id = img_ids[np.random.randint(0, len(img_ids))]
img_info = coco_train.loadImgs(sample_img_id)[0]
image = np.array(Image.open(DATA_DIR / "images" / "Train" / img_info["file_name"]).convert("RGB"))
masks = get_instance_masks(coco_train, sample_img_id)

TRANSFORMS_TO_EXPLORE = {
    "horizontal_flip": A.HorizontalFlip(p=1.0),
    "vertical_flip": A.VerticalFlip(p=1.0),
    "zoom": A.Affine(scale=(0.7, 1.3), p=1.0),
    "clahe_default": A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
    "clahe_strong": A.CLAHE(clip_limit=12.0, tile_grid_size=(8, 8), p=1.0),
    "autocontrast_default": A.AutoContrast(cutoff=0, p=1.0),
    "autocontrast_aggressive": A.AutoContrast(cutoff=15, p=1.0),
    "brightness_contrast": A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
    "hue_saturation_value": A.HueSaturationValue(
        hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=20, p=1.0
    ),
}

for name, transform in TRANSFORMS_TO_EXPLORE.items():
    plot_augmentation_comparison(image, masks, transform, title=name, show_labels=True, seed=0)

### Copy-paste (`A.OverlayElements`)

Albumentations doesn't have a transform literally named "copy-paste", but `OverlayElements` does exactly that: paste a cropped instance (image + its own mask) from one image onto a different base image, at either a given or randomly chosen location, stamping the pasted region into the base's mask at `mask_id`. Cuts one instance from a *donor* image via its bounding box + mask, pastes it onto a different *base* image.

In [ ]:
def get_instance_crop(
    coco: COCO, img_id: int, ann_idx: int, img_dir: Path
) -> tuple[np.ndarray, np.ndarray]:
    """Crop one instance's bounding-box region (image + binary mask) as a copy-paste donor."""
    img_info = coco.loadImgs(img_id)[0]
    image = np.array(Image.open(Path(img_dir) / img_info["file_name"]).convert("RGB"))
    ann = coco.loadAnns(coco.getAnnIds(imgIds=img_id))[ann_idx]
    mask = coco.annToMask(ann)
    x, y, w, h = (int(v) for v in ann["bbox"])
    return image[y : y + h, x : x + w], mask[y : y + h, x : x + w]


def copy_paste_demo(
    coco: COCO,
    base_img_id: int,
    donor_img_id: int,
    donor_ann_idx: int,
    img_dir: Path,
    seed: int | None = None,
) -> None:
    """Paste one instance cut from `donor_img_id` onto `base_img_id`."""
    base_info = coco.loadImgs(base_img_id)[0]
    base_image = np.array(Image.open(Path(img_dir) / base_info["file_name"]).convert("RGB"))
    base_masks = get_instance_masks(coco, base_img_id)
    base_canvas_mask = np.zeros(base_image.shape[:2], dtype=np.uint8)

    donor_crop, donor_mask = get_instance_crop(coco, donor_img_id, donor_ann_idx, img_dir)

    pipeline = A.Compose([A.OverlayElements(p=1.0)])
    if seed is not None:
        pipeline.set_random_seed(seed)
    result = pipeline(
        image=base_image,
        mask=base_canvas_mask,
        overlay_metadata=[{"image": donor_crop, "mask": donor_mask, "mask_id": 1}],
    )
    pasted_image, pasted_mask = result["image"], result["mask"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    axes[0].imshow(donor_crop)
    axes[0].set_title("donor instance (cropped)")
    axes[1].imshow(overlay_masks(base_image, base_masks))
    axes[1].set_title("base image + its own masks")
    axes[2].imshow(overlay_masks(pasted_image, [*base_masks, pasted_mask.astype(bool)]))
    axes[2].set_title("base + pasted donor")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# int(...): np.random.choice returns numpy.int64, and pycocotools' COCO.loadImgs
# silently returns None (not a KeyError) for a numpy int rather than a plain Python
# int -- easy to lose an hour to, found by actually running this end to end.
base_id, donor_id = (int(i) for i in np.random.choice(img_ids, size=2, replace=False))
copy_paste_demo(
    coco_train, base_id, donor_id, donor_ann_idx=0, img_dir=DATA_DIR / "images" / "Train", seed=0
)

### Copy-paste multiple donors, avoiding heavy overlap

Extends the single-donor demo above to paste several random instances onto one base image, rejection-sampling each donor's placement (`find_non_overlapping_offset`) so it doesn't land mostly on top of an instance already there — either an original base instance or an earlier donor pasted this same call. Donors are composited directly with numpy (mask-driven pixel copy) rather than through `A.OverlayElements`, since the placement has to be *chosen* (and bad candidates rejected) before any compositing happens — at which point handing that offset to `OverlayElements` wouldn't do anything simpler than just copying the masked pixels in directly.

In [ ]:
from collections.abc import Callable


def sample_donor(coco: COCO, img_dir: Path, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """Crop one uniformly random instance from a uniformly random image, as a copy-paste donor.

    Args:
        coco: COCO object to sample from.
        img_dir: Directory the raw images live in.
        rng: Random generator to draw the image/instance from.

    Returns:
        Tuple of (donor crop, donor mask), same shape as `get_instance_crop`'s return.
    """
    img_ids = coco.getImgIds()
    donor_img_id = int(rng.choice(img_ids))  # int(): see the np.int64 gotcha noted above
    anns = coco.getAnnIds(imgIds=donor_img_id)
    ann_idx = int(rng.integers(0, len(anns)))
    return get_instance_crop(coco, donor_img_id, ann_idx, img_dir)


def find_non_overlapping_offset(
    occupied_mask: np.ndarray,
    donor_mask: np.ndarray,
    rng: np.random.Generator,
    max_overlap_ratio: float = 0.15,
    max_attempts: int = 30,
) -> tuple[int, int] | None:
    """Rejection-sample a (y, x) top-left offset for `donor_mask` that keeps overlap
    with already-placed instances under `max_overlap_ratio` of the donor's own area.

    Args:
        occupied_mask: Binary canvas (base image's H, W) marking pixels already covered
            by the base image's own instances or previously pasted donors.
        donor_mask: Binary mask of the donor crop (donor crop's own H, W).
        rng: Random generator to draw candidate offsets from.
        max_overlap_ratio: Largest fraction of the donor's own mask area allowed to land
            on already-occupied pixels before an offset is rejected.
        max_attempts: How many random offsets to try before giving up.

    Returns:
        (y, x) top-left offset, or None if no attempt met the overlap budget within
        `max_attempts` -- caller decides whether to skip this donor.
    """
    canvas_h, canvas_w = occupied_mask.shape
    donor_h, donor_w = donor_mask.shape
    donor_area = donor_mask.sum()
    if donor_area == 0 or donor_h > canvas_h or donor_w > canvas_w:
        return None

    for _ in range(max_attempts):
        y = int(rng.integers(0, canvas_h - donor_h + 1))
        x = int(rng.integers(0, canvas_w - donor_w + 1))
        overlap = np.logical_and(occupied_mask[y : y + donor_h, x : x + donor_w], donor_mask).sum()
        if overlap / donor_area <= max_overlap_ratio:
            return y, x
    return None


def crop_to_mask_bbox(image: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Trim `image`/`mask` down to the tight bounding box of `mask`'s nonzero pixels.

    Needed after donor-side spatial augmentation (rotation especially): Albumentations
    keeps its own output canvas, which is usually larger than the actual rotated
    silhouette -- trimming keeps the donor's placement footprint tight instead of
    carrying dead padding around it.

    Args:
        image: RGB image, same H, W as `mask`.
        mask: Binary mask, nonzero where the donor actually is.

    Returns:
        Tuple of (trimmed image, trimmed mask). If `mask` is entirely empty, returns
        both unchanged -- `find_non_overlapping_offset` already treats a zero-area mask
        as unplaceable.
    """
    ys, xs = np.where(mask)
    if ys.size == 0:
        return image, mask
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    return image[y0:y1, x0:x1], mask[y0:y1, x0:x1]


def instance_sizes(masks: list[np.ndarray]) -> np.ndarray:
    """Compute a linear size metric (sqrt of mask area) for each instance mask.

    sqrt(area) rather than raw width/height or box diagonal: comparable across
    instances regardless of aspect ratio or pose, and scales linearly with the bird's
    apparent size the way a resize factor needs to.

    Args:
        masks: List of binary instance masks.

    Returns:
        1D array of sqrt(area) values, one per non-empty mask.
    """
    return np.array([np.sqrt(area) for m in masks if (area := m.astype(bool).sum()) > 0])


def sample_domain_scale_factor(
    donor_mask: np.ndarray,
    target_sizes: np.ndarray,
    rng: np.random.Generator,
    jitter: float = 0.15,
) -> float:
    """Sample a resize factor that brings a donor's size in line with a target scene's
    own instance-size distribution, instead of the donor's native pixel scale (a
    function of *its source image's* camera distance, unrelated to the target scene).

    Models the target scene's sizes as roughly normal: animals arriving at a farm are
    typically similar age/weight, and a normal weight (and so apparent-size) spread
    develops across a flock as it grows, rather than a fixed handful of size classes --
    so the reference size is drawn from `Normal(mean(target_sizes), std(target_sizes))`,
    directly reusing the scene's own observed spread rather than an arbitrary jitter
    range. `jitter` only kicks in as a floor on that std, for scenes with too few
    instances to estimate spread reliably (or a std of exactly 0, e.g. one instance).

    Args:
        donor_mask: The donor's current binary mask (before resizing).
        target_sizes: Reference sizes (sqrt(area), see `instance_sizes`) drawn from the
            target scene's own instances.
        rng: Random generator for the reference draw.
        jitter: Minimum reference-size spread, as a fraction of the mean -- a floor
            under `target_sizes`'s own std, not the primary source of variance.

    Returns:
        Scale factor to resize the donor by (1.0 = no resize). Returns 1.0 if
        `target_sizes` is empty (nothing to match against) or the donor mask is empty.
    """
    donor_size = np.sqrt(donor_mask.astype(bool).sum())
    if donor_size == 0 or target_sizes.size == 0:
        return 1.0

    mean_size = float(np.mean(target_sizes))
    std_size = max(float(np.std(target_sizes)), jitter * mean_size)  # floor: too-few-samples std isn't trustworthy
    reference_size = max(rng.normal(mean_size, std_size), 0.25 * mean_size)  # keep the normal's tail sane
    return float(reference_size / donor_size)


def resize_donor(image: np.ndarray, mask: np.ndarray, scale_factor: float) -> tuple[np.ndarray, np.ndarray]:
    """Resize a donor crop + mask by `scale_factor`.

    Args:
        image: Donor RGB crop.
        mask: Donor binary mask, same H, W as `image`.
        scale_factor: Multiplicative resize factor (1.0 = unchanged). Floored so a
            pathological factor can't collapse the donor below a few pixels.

    Returns:
        Tuple of (resized image, resized mask). Mask uses nearest-neighbor
        interpolation to stay binary; image uses area interpolation when shrinking and
        linear when enlarging (cv2's usual recommendation for each direction).
    """
    h, w = mask.shape
    scale_factor = max(scale_factor, 4 / max(h, w, 1))  # floor: keep at least ~4px on the long side
    new_w, new_h = max(1, round(w * scale_factor)), max(1, round(h * scale_factor))

    interp = cv2.INTER_AREA if scale_factor < 1 else cv2.INTER_LINEAR
    resized_image = cv2.resize(image, (new_w, new_h), interpolation=interp)
    resized_mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    return resized_image, resized_mask


def masked_pixel_stats(image: np.ndarray, masks: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """Per-channel LAB mean and std of an image's pixels, pooled across instance masks.

    Pools pixels from every mask together rather than averaging per-instance stats --
    within one scene, all real birds share roughly the same lighting condition, so the
    combined pixel population is the more direct signal than an average-of-averages.
    LAB (not RGB) separates lightness from chrominance, so a brightness mismatch and a
    color-cast mismatch (e.g. a bluish tint) both show up as a clean per-channel shift
    instead of an entangled RGB one.

    Args:
        image: RGB image the masks index into.
        masks: One binary mask per instance to pool pixels from.

    Returns:
        Tuple of (per-channel LAB mean, per-channel LAB std), each a length-3 float
        array. Both default to (0, 1) if no mask has any nonzero pixels -- callers
        should treat that as "no signal," same convention as
        `sample_domain_scale_factor`'s empty-`target_sizes` case.
    """
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB).astype(np.float32)
    pooled = [lab[m.astype(bool)] for m in masks if m.astype(bool).any()]
    if not pooled:
        return np.zeros(3, dtype=np.float32), np.ones(3, dtype=np.float32)
    pixels = np.concatenate(pooled, axis=0)
    return pixels.mean(axis=0), pixels.std(axis=0)


def match_color_to_target(
    donor_image: np.ndarray,
    donor_mask: np.ndarray,
    target_mean: np.ndarray,
    target_std: np.ndarray,
    strength: float = 1.0,
    std_floor: float = 5.0,
) -> np.ndarray:
    """Shift/scale a donor's colors toward a target LAB mean+std (Reinhard-style color transfer).

    Same shape of idea as `sample_domain_scale_factor` for size: reuse the target
    scene's own observed color statistics (from `masked_pixel_stats`) instead of an
    arbitrary fixed correction, since a donor pulled from a different facility/lighting
    condition needs a scene-specific correction, not a universal one.

    Args:
        donor_image: Donor RGB crop (whole crop, not just masked pixels -- only mask
            pixels get pasted downstream, so recoloring the rest of the crop is harmless).
        donor_mask: The donor's own binary mask, used to compute *its* current stats.
        target_mean: Target per-channel LAB mean, from `masked_pixel_stats`.
        target_std: Target per-channel LAB std, from `masked_pixel_stats`.
        strength: 1.0 = full statistical match, 0.0 = no change, in between blends.
        std_floor: Minimum donor std per channel before dividing by it -- guards a
            near-solid-color donor crop from an exploding scale factor.

    Returns:
        A recolored copy of `donor_image`, same shape and dtype (uint8).
    """
    region = donor_mask.astype(bool)
    if not region.any():
        return donor_image

    lab = cv2.cvtColor(donor_image, cv2.COLOR_RGB2LAB).astype(np.float32)
    donor_mean = lab[region].mean(axis=0)
    donor_std = np.maximum(lab[region].std(axis=0), std_floor)

    matched = (lab - donor_mean) * (target_std / donor_std) + target_mean
    blended = lab + strength * (matched - lab)
    blended_uint8 = np.clip(blended, 0, 255).astype(np.uint8)
    return cv2.cvtColor(blended_uint8, cv2.COLOR_LAB2RGB)


def copy_paste_compose(
    base_image: np.ndarray,
    base_masks: list[np.ndarray],
    donor_fn: Callable[[], tuple[np.ndarray, np.ndarray]],
    rng: np.random.Generator,
    n_donors: int,
    donor_augment: A.Compose | None = None,
    target_sizes: np.ndarray | None = None,
    size_jitter: float = 0.15,
    target_color_stats: tuple[np.ndarray, np.ndarray] | None = None,
    color_strength: float = 1.0,
    max_overlap_ratio: float = 0.15,
    max_attempts: int = 30,
) -> tuple[np.ndarray, list[np.ndarray], int]:
    """Paste up to `n_donors` donors (drawn from `donor_fn`) onto `base_image`,
    rejecting placements that overlap too much with what's already there.

    Shared compositing core behind `copy_paste_multi_demo` (live-COCO donors) and
    `copy_paste_bank_demo` (curated-bank donors) -- the two differ only in *where*
    donors come from, not in how augmentation/color/resizing/placement/compositing
    works. Each donor goes through, in order: `donor_augment` (orientation) -> color
    match toward `target_color_stats` -> resize toward `target_sizes` (scale) ->
    `find_non_overlapping_offset` (placement).

    Args:
        base_image: RGB image to paste onto.
        base_masks: The base image's own instance masks -- the starting occupied region.
        donor_fn: Zero-arg callable returning one (donor crop, donor mask) pair per call.
        rng: Random generator, passed through to `find_non_overlapping_offset` and
            `sample_domain_scale_factor`.
        n_donors: How many donors to attempt to paste.
        donor_augment: Optional Albumentations pipeline (`image`/`mask` targets) applied
            to each donor -- e.g. flip/rotate -- before color/resizing/placement. Each
            donor is re-cropped to its transformed mask's bounding box afterward via
            `crop_to_mask_bbox`. `None` skips augmentation entirely.
        target_sizes: Optional reference sizes (sqrt(area), see `instance_sizes`) to
            resize each donor toward via `sample_domain_scale_factor` -- e.g.
            `instance_sizes(base_masks)` to match the base image's own instances.
            `None` skips resizing (donor keeps whatever size augmentation left it at).
        size_jitter: See `sample_domain_scale_factor`.
        target_color_stats: Optional `(mean, std)` LAB pair (see `masked_pixel_stats`)
            to color-match each donor toward via `match_color_to_target` -- e.g.
            `masked_pixel_stats(base_image, base_masks)` to match the base image's own
            instances. `None` skips color matching (donor keeps its native colors).
        color_strength: See `match_color_to_target`.
        max_overlap_ratio: See `find_non_overlapping_offset`.
        max_attempts: See `find_non_overlapping_offset`.

    Returns:
        Tuple of (composed image, list of pasted donor masks, number of donors skipped
        because no placement met the overlap budget within `max_attempts`).
    """
    occupied = np.zeros(base_image.shape[:2], dtype=bool)
    for m in base_masks:
        occupied |= m.astype(bool)

    composed_image = base_image.copy()
    pasted_masks = []
    skipped = 0

    for _ in range(n_donors):
        donor_crop, donor_mask = donor_fn()

        if donor_augment is not None:
            augmented = donor_augment(image=donor_crop, mask=donor_mask)
            donor_crop, donor_mask = crop_to_mask_bbox(augmented["image"], augmented["mask"])

        if target_color_stats is not None:
            target_mean, target_std = target_color_stats
            donor_crop = match_color_to_target(donor_crop, donor_mask, target_mean, target_std, strength=color_strength)

        if target_sizes is not None and target_sizes.size > 0:
            scale_factor = sample_domain_scale_factor(donor_mask, target_sizes, rng, jitter=size_jitter)
            donor_crop, donor_mask = resize_donor(donor_crop, donor_mask, scale_factor)

        donor_mask = donor_mask.astype(bool)
        offset = find_non_overlapping_offset(occupied, donor_mask, rng, max_overlap_ratio, max_attempts)
        if offset is None:
            skipped += 1
            continue

        y, x = offset
        h, w = donor_mask.shape
        region = (slice(y, y + h), slice(x, x + w))
        composed_image[region][donor_mask] = donor_crop[donor_mask]

        full_mask = np.zeros(base_image.shape[:2], dtype=bool)
        full_mask[region] = donor_mask
        occupied |= full_mask
        pasted_masks.append(full_mask)

    return composed_image, pasted_masks, skipped


def copy_paste_multi_demo(
    coco: COCO,
    base_img_id: int,
    img_dir: Path,
    n_donors: int = 4,
    donor_augment: A.Compose | None = None,
    match_target_scale: bool = False,
    size_jitter: float = 0.15,
    match_target_color: bool = False,
    color_strength: float = 1.0,
    max_overlap_ratio: float = 0.15,
    max_attempts: int = 30,
    seed: int | None = None,
    show_masks: bool = True,
) -> None:
    """Paste several random donor instances onto `base_img_id`, rejecting placements
    that would overlap too much with what's already there.

    Donors are drawn uniformly at random across all of `coco` (any image, any
    instance) -- including `base_img_id` itself, which is a legitimate copy-paste case
    (denser occlusion of the same bird), not excluded. See `copy_paste_bank_demo` below
    for the curated-bank equivalent.

    Args:
        coco: COCO object to draw both the base image and donor instances from.
        base_img_id: Image id to paste onto.
        img_dir: Directory the raw images live in.
        n_donors: How many donor instances to attempt to paste.
        donor_augment: See `copy_paste_compose`. Seeded from `seed` here if both are set.
        match_target_scale: If True, resize each donor toward `base_img_id`'s own
            instance-size distribution (`instance_sizes(base_masks)`) via
            `copy_paste_compose`'s `target_sizes` -- see `sample_domain_scale_factor`.
        size_jitter: See `sample_domain_scale_factor`.
        match_target_color: If True, color-match each donor toward `base_img_id`'s own
            instance color statistics (`masked_pixel_stats(base_image, base_masks)`) via
            `copy_paste_compose`'s `target_color_stats` -- see `match_color_to_target`.
        color_strength: See `match_color_to_target`.
        max_overlap_ratio: See `find_non_overlapping_offset`.
        max_attempts: See `find_non_overlapping_offset`.
        seed: Seeds donor sampling, placement, and `donor_augment`, for a reproducible draw.
        show_masks: If True (default), draw the usual semi-transparent mask overlay on
            both panels. If False, show raw pixels instead -- masks make instance
            separation legible but also tint over a donor's true color, which hides
            exactly the effect `match_target_color` is meant to show.
    """
    if donor_augment is not None and seed is not None:
        donor_augment.set_random_seed(seed)
    rng = np.random.default_rng(seed)

    base_info = coco.loadImgs(base_img_id)[0]
    base_image = np.array(Image.open(Path(img_dir) / base_info["file_name"]).convert("RGB"))
    base_masks = get_instance_masks(coco, base_img_id)
    target_sizes = instance_sizes(base_masks) if match_target_scale else None
    target_color_stats = masked_pixel_stats(base_image, base_masks) if match_target_color else None

    composed_image, pasted_masks, skipped = copy_paste_compose(
        base_image,
        base_masks,
        donor_fn=lambda: sample_donor(coco, img_dir, rng),
        rng=rng,
        n_donors=n_donors,
        donor_augment=donor_augment,
        target_sizes=target_sizes,
        size_jitter=size_jitter,
        target_color_stats=target_color_stats,
        color_strength=color_strength,
        max_overlap_ratio=max_overlap_ratio,
        max_attempts=max_attempts,
    )

    if skipped:
        print(
            f"Skipped {skipped}/{n_donors} donor(s): no placement stayed under "
            f"{max_overlap_ratio:.0%} overlap after {max_attempts} attempts each."
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    left = overlay_masks(base_image, base_masks) if show_masks else base_image
    right = overlay_masks(composed_image, [*base_masks, *pasted_masks]) if show_masks else composed_image
    axes[0].imshow(left)
    axes[0].set_title(f"base image + its own masks ({len(base_masks)} instances)")
    axes[1].imshow(right)
    axes[1].set_title(f"base + {len(pasted_masks)} pasted donor(s) (live-sampled)")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Reuses base_id from the single-donor demo above so the two are easy to compare on the
# same base image.
copy_paste_multi_demo(
    coco_train, base_id, img_dir=DATA_DIR / "images" / "Train", n_donors=6, seed=0
)

### Towards synthetic training data: a curated, disk-cached donor bank

`sample_donor` above is uniformly random across *all* instances, which on a densely-occluded overhead dataset like ChickenVerse frequently grabs a half-visible bird as a donor -- pasting a partially-occluded fragment onto a new image looks obviously wrong, not like added density. `find_unoccluded_untruncated_donor_instances` filters to donors whose mask fills most of its own bounding box (not occluded) and whose bounding box doesn't touch the image edge (not cut off), reusing the same mask/box area-ratio signal already computed in the Mask Quality Checks section above.

`build_donor_bank` then materializes a curated subset's crops + masks to disk once. Storage is one `.png` image + one `.png` mask per donor (not a single `.npz`, and not COCO/YOLO-label format): plain PNGs mean the bank is browsable in any image viewer -- useful for eyeballing curation quality directly, which is exactly how the thresholds below got tuned -- and lossless, unlike round-tripping the mask through a YOLO-style polygon label (this project already paid for that lossy detour once, see the RLE-to-polygon section above; no reason to reintroduce it here when the raster mask is already in hand). COCO's per-image/per-annotation json structure doesn't fit either: it's built for full scenes with many annotations, not a flat bank of independent single-instance crops. Cached under `DATA_DIR/copy_paste_donor_bank/`, same persistent-cache pattern as `annotations_polygon_cache/` earlier, so later cells -- or a real training run -- sample directly from a pre-filtered pool instead of re-scanning and re-filtering the whole COCO index every time.

In [ ]:
def find_unoccluded_untruncated_donor_instances(
    coco: COCO,
    min_area_ratio: float = 0.6,
    min_mask_area: int = 70,
    border_margin: int = 80,
) -> list[dict]:
    """Find COCO annotations that make good copy-paste donors: not occluded, not
    truncated at the image border, and not tiny slivers.

    Uses `mask_utils.area` directly on the RLE (no full pixel decode), the same
    efficient pattern `check_mask_quality` already uses, so scanning every annotation
    in a split stays fast.

    Args:
        coco: COCO object to scan.
        min_area_ratio: Minimum mask-area / box-area ratio to accept -- near 1 means
            the visible mask fills its own bounding box, i.e. not occluded/cut off.
            Tuned to 0.6, deliberately *below* the dataset's own observed median
            (0.64-0.68 across splits, per Mask Quality Checks above): moderate self-/
            mutual occlusion is the typical case for a ChickenVerse instance, not an
            exception, so a threshold above the median would reject most of the
            population instead of just the genuinely badly-occluded tail.
        min_mask_area: Minimum mask area in pixels, to skip near-invisible slivers.
        border_margin: Pixels of tolerance before a bbox touching the image edge is
            treated as truncated and rejected. Deliberately large -- 80px, ~8% of this
            dataset's 1024px frame width -- a quality-over-quantity choice: a small
            margin (just a few px) is already enough to catch genuinely truncated
            instances, since a cut-off bird's visible silhouette naturally extends
            right up to the frame edge with nothing to gain from a bigger buffer *for
            that purpose*. The extra margin here is insurance against fully-visible but
            near-edge instances that could still catch lens vignetting/distortion --
            worth it because the candidate pool stays abundant even at this size (tens
            of thousands of instances, only 500 needed for the bank), so there's no real
            cost to being conservative, unlike `min_area_ratio` above.

    Returns:
        List of `{"image_id", "ann_id"}` dicts for each accepted instance -- lightweight
        enough to keep in memory; the actual crop/mask is materialized lazily.
    """
    candidates = []
    for img_id in coco.getImgIds():
        img_info = coco.loadImgs(img_id)[0]
        img_w, img_h = img_info["width"], img_info["height"]

        for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
            seg = ann.get("segmentation")
            if seg is None or (isinstance(seg, list) and len(seg) == 0):
                continue

            x, y, w, h = ann["bbox"]
            if x <= border_margin or y <= border_margin:
                continue
            if x + w >= img_w - border_margin or y + h >= img_h - border_margin:
                continue

            if isinstance(seg, list):
                rle = mask_utils.frPyObjects(seg, img_h, img_w)
                mask_area = float(mask_utils.area(rle).sum())
            elif isinstance(seg, dict):
                rle = mask_utils.frPyObjects([seg], seg["size"][0], seg["size"][1])
                mask_area = float(mask_utils.area(rle)[0])
            else:
                continue

            box_area = w * h
            if mask_area < min_mask_area or box_area <= 0 or mask_area / box_area < min_area_ratio:
                continue

            candidates.append({"image_id": img_id, "ann_id": ann["id"]})
    return candidates


def build_donor_bank(
    coco: COCO,
    img_dir: Path,
    bank_dir: Path,
    max_donors: int = 500,
    min_area_ratio: float = 0.6,
    min_mask_area: int = 70,
    border_margin: int = 80,
    seed: int | None = 0,
) -> list[dict]:
    """Curate donor instances and cache their crop + mask to disk as a reusable bank.

    Scans `coco` via `find_unoccluded_untruncated_donor_instances`, takes a random
    subset (up to `max_donors`), and writes each donor's cropped image as
    `<donor_id>.png` and its mask as `<donor_id>_mask.png` (single-channel, 0/255)
    under `bank_dir`, plus a `manifest.json` describing the bank.

    Args:
        coco: COCO object to curate donors from.
        img_dir: Directory the raw images live in.
        bank_dir: Where to write the bank (created if missing).
        max_donors: Cap on how many curated donors to materialize to disk.
        min_area_ratio: See `find_unoccluded_untruncated_donor_instances`.
        min_mask_area: See `find_unoccluded_untruncated_donor_instances`.
        border_margin: See `find_unoccluded_untruncated_donor_instances`.
        seed: Seeds the random subset chosen from the full candidate pool.

    Returns:
        The manifest list (also written to `bank_dir/manifest.json`): one dict per
        donor with `donor_id`, `category_id`, `h`, `w`.
    """
    bank_dir.mkdir(parents=True, exist_ok=True)

    candidates = find_unoccluded_untruncated_donor_instances(coco, min_area_ratio, min_mask_area, border_margin)
    print(
        f"{len(candidates)} unoccluded, untruncated candidate instance(s) found "
        f"(of {len(coco.getAnnIds())} total)."
    )

    rng = np.random.default_rng(seed)
    n_take = min(max_donors, len(candidates))
    chosen_idx = rng.choice(len(candidates), size=n_take, replace=False)

    manifest = []
    for i in chosen_idx:
        c = candidates[int(i)]
        img_info = coco.loadImgs(c["image_id"])[0]
        ann = coco.loadAnns([c["ann_id"]])[0]
        image = np.array(Image.open(img_dir / img_info["file_name"]).convert("RGB"))
        mask = decode_rle_mask(coco, ann)
        x, y, w, h = (int(v) for v in ann["bbox"])
        crop, crop_mask = image[y : y + h, x : x + w], mask[y : y + h, x : x + w]

        donor_id = f"ann{c['ann_id']}"
        Image.fromarray(crop).save(bank_dir / f"{donor_id}.png")
        Image.fromarray((crop_mask.astype(np.uint8)) * 255).save(bank_dir / f"{donor_id}_mask.png")
        manifest.append({"donor_id": donor_id, "category_id": ann["category_id"], "h": h, "w": w})

    (bank_dir / "manifest.json").write_text(json.dumps(manifest))
    print(f"Donor bank: {len(manifest)} donor(s) written to {bank_dir}")
    return manifest


def load_donor_bank(bank_dir: Path) -> list[dict]:
    """Load a donor bank's manifest, written by `build_donor_bank`."""
    manifest_path = bank_dir / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"No donor bank manifest at {manifest_path} -- run build_donor_bank first.")
    return json.loads(manifest_path.read_text())


def sample_donor_from_bank(
    bank_dir: Path, manifest: list[dict], rng: np.random.Generator
) -> tuple[np.ndarray, np.ndarray]:
    """Sample one donor crop + mask from a pre-built bank on disk.

    Args:
        bank_dir: Directory `build_donor_bank` wrote the bank to.
        manifest: Bank manifest, from `build_donor_bank`/`load_donor_bank`.
        rng: Random generator to pick the donor with.

    Returns:
        Tuple of (donor crop, donor mask) -- a drop-in replacement for `sample_donor`'s
        return shape.
    """
    entry = manifest[int(rng.integers(0, len(manifest)))]
    crop = np.array(Image.open(bank_dir / f"{entry['donor_id']}.png").convert("RGB"))
    mask = (np.array(Image.open(bank_dir / f"{entry['donor_id']}_mask.png").convert("L")) > 0).astype(np.uint8)
    return crop, mask

In [ ]:
DONOR_BANK_DIR = DATA_DIR / "copy_paste_donor_bank"  # persistent cache, not scratch -- same pattern as POLYGON_ANN_DIR above
FORCE_REBUILD_DONOR_BANK = False  # flip to True after changing the curation filters above

if DONOR_BANK_DIR.joinpath("manifest.json").exists() and not FORCE_REBUILD_DONOR_BANK:
    donor_manifest = load_donor_bank(DONOR_BANK_DIR)
    print(f"Using cached donor bank: {len(donor_manifest)} donor(s) at {DONOR_BANK_DIR}")
else:
    donor_manifest = build_donor_bank(
        coco_train, DATA_DIR / "images" / "Train", DONOR_BANK_DIR, max_donors=500, seed=0
    )

In [ ]:
def plot_donor_bank_sample(bank_dir: Path, manifest: list[dict], n: int = 8, seed: int | None = 0) -> None:
    """Show a grid of donor crops from the bank, background dimmed outside the mask --
    a quick eyeball check that `find_unoccluded_untruncated_donor_instances`'s filters
    are actually picking unoccluded, untruncated birds rather than random fragments.
    """
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(manifest), size=min(n, len(manifest)), replace=False)

    cols = 4
    rows = -(-len(idx) // cols)  # ceil division
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, i in zip(axes, idx):
        entry = manifest[int(i)]
        crop = np.array(Image.open(bank_dir / f"{entry['donor_id']}.png").convert("RGB"))
        mask = np.array(Image.open(bank_dir / f"{entry['donor_id']}_mask.png").convert("L")) > 0
        display_crop = crop.copy()
        display_crop[~mask] = display_crop[~mask] // 3  # dim the background so the mask boundary is obvious
        ax.imshow(display_crop)
        ax.axis("off")
        ax.set_title(entry["donor_id"], fontsize=8)
    for ax in axes[len(idx) :]:
        ax.axis("off")

    fig.suptitle(f"{len(manifest)}-donor bank -- {len(idx)} random sample(s)")
    plt.tight_layout()
    plt.show()


plot_donor_bank_sample(DONOR_BANK_DIR, donor_manifest, n=8, seed=1)

### Copy-paste from the curated bank

Same placement/overlap-avoidance core as the live-sampled demo above (`copy_paste_compose`, shared) -- only the donor source changes, via `sample_donor_from_bank` instead of `sample_donor`. Worth comparing side by side against the live version on the same base image: the bank version should visibly stop producing half-visible/occluded donors, since those were filtered out when the bank was built.

In [ ]:
def copy_paste_bank_demo(
    coco: COCO,
    base_img_id: int,
    img_dir: Path,
    bank_dir: Path,
    manifest: list[dict],
    n_donors: int = 4,
    donor_augment: A.Compose | None = None,
    match_target_scale: bool = False,
    size_jitter: float = 0.15,
    match_target_color: bool = False,
    color_strength: float = 1.0,
    max_overlap_ratio: float = 0.15,
    max_attempts: int = 30,
    seed: int | None = None,
    show_masks: bool = True,
) -> None:
    """Paste several donors from the curated bank onto `base_img_id`.

    Same placement logic as `copy_paste_multi_demo` (via the shared `copy_paste_compose`
    core), but donors come from `sample_donor_from_bank` -- pre-filtered for occlusion
    and border-truncation when the bank was built -- instead of a live, unfiltered scan.

    Args:
        coco: COCO object the base image's own masks are drawn from.
        base_img_id: Image id to paste onto.
        img_dir: Directory the raw images live in.
        bank_dir: Directory `build_donor_bank` wrote the bank to.
        manifest: Bank manifest, from `build_donor_bank`/`load_donor_bank`.
        n_donors: How many donor instances to attempt to paste.
        donor_augment: See `copy_paste_compose`. Seeded from `seed` here if both are set.
        match_target_scale: If True, resize each donor toward `base_img_id`'s own
            instance-size distribution -- see `copy_paste_multi_demo`.
        size_jitter: See `sample_domain_scale_factor`.
        match_target_color: If True, color-match each donor toward `base_img_id`'s own
            instance color statistics -- see `copy_paste_multi_demo`.
        color_strength: See `match_color_to_target`.
        max_overlap_ratio: See `find_non_overlapping_offset`.
        max_attempts: See `find_non_overlapping_offset`.
        seed: Seeds donor sampling, placement, and `donor_augment`, for a reproducible draw.
        show_masks: See `copy_paste_multi_demo`.
    """
    if donor_augment is not None and seed is not None:
        donor_augment.set_random_seed(seed)
    rng = np.random.default_rng(seed)

    base_info = coco.loadImgs(base_img_id)[0]
    base_image = np.array(Image.open(Path(img_dir) / base_info["file_name"]).convert("RGB"))
    base_masks = get_instance_masks(coco, base_img_id)
    target_sizes = instance_sizes(base_masks) if match_target_scale else None
    target_color_stats = masked_pixel_stats(base_image, base_masks) if match_target_color else None

    composed_image, pasted_masks, skipped = copy_paste_compose(
        base_image,
        base_masks,
        donor_fn=lambda: sample_donor_from_bank(bank_dir, manifest, rng),
        rng=rng,
        n_donors=n_donors,
        donor_augment=donor_augment,
        target_sizes=target_sizes,
        size_jitter=size_jitter,
        target_color_stats=target_color_stats,
        color_strength=color_strength,
        max_overlap_ratio=max_overlap_ratio,
        max_attempts=max_attempts,
    )

    if skipped:
        print(
            f"Skipped {skipped}/{n_donors} donor(s): no placement stayed under "
            f"{max_overlap_ratio:.0%} overlap after {max_attempts} attempts each."
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    left = overlay_masks(base_image, base_masks) if show_masks else base_image
    right = overlay_masks(composed_image, [*base_masks, *pasted_masks]) if show_masks else composed_image
    axes[0].imshow(left)
    axes[0].set_title(f"base image + its own masks ({len(base_masks)} instances)")
    axes[1].imshow(right)
    axes[1].set_title(f"base + {len(pasted_masks)} pasted donor(s) (from bank)")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# Reuses base_id so this is directly comparable against the live-sampled version above.
copy_paste_bank_demo(
    coco_train, base_id, DATA_DIR / "images" / "Train", DONOR_BANK_DIR, donor_manifest, n_donors=6, seed=0
)

### Donor-side augmentation before placement

Each donor gets its own independent flip/rotate draw (`donor_augment`, run inside `copy_paste_compose` via the `image`/`mask` targets) before placement is attempted -- so pasting the same bank donor twice doesn't look identical twice. Full `rotate=(-180, 180)` is deliberate, not a small jitter: this is overhead imagery with no canonical "up" for a bird (same reasoning as `VerticalFlip` in the augmentation section above), so any orientation is plausible. `fit_output=True` keeps `Affine` from clipping the donor when it rotates -- without it, Albumentations keeps the *input* canvas size, and a bird rotated near 45 degrees would get its corners cut off by the crop's original bounding box. `crop_to_mask_bbox` (in the cell above) then trims the now-larger canvas back down to the transformed silhouette's own tight bounding box, so placement isn't working with a mostly-empty canvas.

Scale is deliberately *not* in this pipeline -- that's handled by the dedicated domain-aware resize step below (`sample_domain_scale_factor`/`resize_donor`), which matches the donor's size to the *target* scene's own instances rather than an arbitrary jitter range. Keeping orientation and scale as separate, independently composable steps (rather than one `Affine` doing both) is also why `resize_donor` is its own atomic function taking a plain `scale_factor` -- easy to call directly outside this exploration flow later.

In [ ]:
DONOR_AUGMENT = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Affine(rotate=(-180, 180), fit_output=True, p=1.0),  # orientation only -- scale is a separate step below
    ]
)

# Reuses base_id again -- directly comparable against the non-augmented bank demo above.
copy_paste_bank_demo(
    coco_train,
    base_id,
    DATA_DIR / "images" / "Train",
    DONOR_BANK_DIR,
    donor_manifest,
    n_donors=6,
    donor_augment=DONOR_AUGMENT,
    seed=0,
)

### Domain-aware resize: matching the target scene's own scale

`instance_sizes` reduces a list of masks to one sqrt(area) number per instance. `sample_domain_scale_factor` fits a Normal(mean, std) to a target scene's own instance sizes and draws the reference from it -- not a single bootstrapped individual, and not an arbitrary fixed jitter, but the scene's own empirical mean *and* spread. This matches how animal cohorts actually work in a farm setting: a flock arrives roughly age/weight-matched and fans out into a real weight (and so apparent-size) distribution as it grows, so a tightly-clustered young flock should produce tightly-sized synthetic birds, and a more size-diverse later-stage flock should produce more size-diverse ones -- reusing whatever spread that particular scene already shows rather than guessing at one fixed number. The reference is relative to the donor's *current* size on purpose, so it composes correctly whether or not `donor_augment` already changed the donor's footprint (e.g. `fit_output`'s canvas growth from rotation). `resize_donor` then does the actual resize, as a plain `(image, mask, scale_factor) -> (image, mask)` function with no dependency on COCO, Albumentations, or anything else in this notebook -- the atomic unit meant to survive a move into `augmentation/segmentation.py` mostly unchanged.

Below: the same base image and donor bank as the orientation-only demo above, now with `match_target_scale=True` added on top -- donors should visibly stop looking randomly-sized relative to the birds already in the scene.

In [ ]:
# Reuses base_id/DONOR_AUGMENT once more -- the only new thing here is match_target_scale=True.
copy_paste_bank_demo(
    coco_train,
    base_id,
    DATA_DIR / "images" / "Train",
    DONOR_BANK_DIR,
    donor_manifest,
    n_donors=6,
    donor_augment=DONOR_AUGMENT,
    match_target_scale=True,
    seed=0,
)

### Color-aware compositing: matching the target scene's lighting

Spotted by eyeballing the raw (non-overlaid) composite while putting together the README example for this feature: pasted donors can carry a visibly different color cast than the scene they land in — ChickenVerse spans 5 facilities with different lighting, so a donor pulled from one facility's conditions doesn't automatically match another's.

`masked_pixel_stats` pools *only the real bird pixels* in the target scene (not the background — the litter is a completely different color from the birds) into a per-channel LAB mean + std. `match_color_to_target` then shifts/scales a donor's own LAB statistics onto that target — the same "reuse the scene's own real distribution" idea as the domain-aware resize above, just for color instead of size. LAB, not RGB: it separates lightness from color, so both a brightness mismatch and a color-cast mismatch (like a bluish tint) get corrected as one clean shift instead of three entangled RGB channels fighting each other.

Shown here **without** the mask overlay, unlike the rest of this section — a semi-transparent mask tint would hide the exact thing being checked.

In [ ]:
base_id

In [ ]:
# Same seed both times -- color matching doesn't consume any randomness, so this draws
# the exact same donors/placements/sizes in both calls, differing only in color.
print("Without color matching:")
copy_paste_bank_demo(
    coco_train,
    base_id,
    DATA_DIR / "images" / "Train",
    DONOR_BANK_DIR,
    donor_manifest,
    n_donors=8,
    donor_augment=DONOR_AUGMENT,
    match_target_scale=True,
    seed=0,
    show_masks=False,
)

print("With color matching:")
copy_paste_bank_demo(
    coco_train,
    base_id,
    DATA_DIR / "images" / "Train",
    DONOR_BANK_DIR,
    donor_manifest,
    n_donors=8,
    donor_augment=DONOR_AUGMENT,
    match_target_scale=True,
    match_target_color=True,
    seed=0,
    show_masks=False,
)

In [ ]:
# lets test different images to see how the color matching works across different lighting conditions
#choose 5 random images from the train set
base_id_list = [int(x) for x in np.random.choice(img_ids, size=5, replace=False)]
print(base_id_list)

for base_demo_id in base_id_list:
    print(f"Base image id: {base_demo_id}")
    print("Without color matching:")
    copy_paste_bank_demo(
        coco_train,
        base_demo_id,
        DATA_DIR / "images" / "Train",
        DONOR_BANK_DIR,
        donor_manifest,
        n_donors=8,
        donor_augment=DONOR_AUGMENT,
        match_target_scale=True,
        match_target_color=False,
        seed=0,
        show_masks=False,
    )
    print("With color matching:")
    copy_paste_bank_demo(
        coco_train,
        base_demo_id,
        DATA_DIR / "images" / "Train",
        DONOR_BANK_DIR,
        donor_manifest,
        n_donors=8,
        donor_augment=DONOR_AUGMENT,
        match_target_scale=True,
        match_target_color=True,
        seed=0,
        show_masks=False,
    )

## Summary — what this notebook found

The short version of everything above, with charts. Full code and reasoning are in the sections you just scrolled past — this is the skim version, for future-me and anyone else picking this notebook up later.

### 1. Mask quality

All three splits are 100% compressed RLE — zero native polygons, and zero missing or degenerate masks. Masks typically cover only 64–68% of their own bounding box, not the full box — expected for a dense, high-occlusion dataset, and the reason `convert_coco` needed help (next section).

In [ ]:
mask_quality_by_split = {name: check_mask_quality(coco) for name, coco in SPLITS.items()}
splits = list(mask_quality_by_split.keys())
medians = [np.median(mask_quality_by_split[s]["mask_to_box_area_ratio"]) for s in splits]

CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a"]  # dataviz skill: fixed categorical order, slots 1-3

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(splits, medians, color=CATEGORICAL[: len(splits)], width=0.5, zorder=3)
ax.set_ylim(0, 1)
ax.set_ylabel("Median mask / box area")
ax.set_title("How much of its own box does a mask actually fill?")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
for bar, v in zip(bars, medians):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", color="#0b0b0b")
plt.tight_layout()
plt.show()

### 2. COCO → YOLO-seg conversion fidelity

`ultralytics.data.converter.convert_coco` can't read compressed RLE — it silently falls back to a box-shaped mask. On a 22-instance test image that drags mean IoU (vs. the real mask) down to 0.63. Converting RLE to polygons first, then feeding *that* to `convert_coco`, brings it back to 0.97.

One thing this chart can't show: only 1 annotation (of 116,329 in Train) had no contour OpenCV could extract from its mask. It's logged by name, not silently dropped — see the RLE→polygon section above.

In [ ]:
STATUS_CRITICAL = "#d03b3b"  # dataviz skill: fixed status palette, not themed
STATUS_GOOD = "#0ca30c"

labels = ["naive convert_coco\n(bbox fallback)", "RLE→polygon\npreprocessing"]
values = [low_fidelity_ious.mean(), high_fidelity_ious.mean()]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, values, color=[STATUS_CRITICAL, STATUS_GOOD], width=0.5, zorder=3)
ax.set_ylim(0, 1)
ax.set_ylabel("Mean mask IoU vs. original COCO mask")
ax.set_title(f"Conversion fidelity — image {sample_img_id} ({len(high_fidelity_ious)} instances)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", color="#0b0b0b")
plt.tight_layout()
plt.show()

### 3. Synthetic copy-paste for training data

Built end-to-end and confirmed working:

- **Curated donor bank** — filters out occluded/cut-off birds *before* caching, not after. 500 donors cached to `.png` pairs on disk.
- **Overlap-aware placement** — rejection-sampled, so a paste doesn't just bury a bird that's already there.
- **Donor-side augmentation** — flip + full 360° rotation (no canonical "up" in overhead shots).
- **Domain-aware resize** — each pasted bird's size is drawn from *that scene's own* size distribution (mean + std), not a fixed range — mirrors how a real flock's weight spread grows over time.

In [ ]:
donor_sizes = np.array([np.sqrt(d["h"] * d["w"]) for d in donor_manifest])

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(donor_sizes, bins=20, color="#2a78d6", edgecolor="#fcfcfb", zorder=3)
ax.set_xlabel("Donor size — sqrt(box area), px")
ax.set_ylabel("Donor count")
ax.set_title(f"Curated donor bank — {len(donor_manifest)} donors")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
plt.tight_layout()
plt.show()

> **Production idea to revisit:** per-image std is a small-sample estimate (often just single-to-low-double-digit birds per image). If realism ever demands it, that std could become a parameter tracked per camera installation, or conditioned on flock age/production stage, instead of recomputed from one image's own noisy sample every time. Not needed yet — flagging for whenever this moves into `augmentation/segmentation.py`.

### 4. Known limitation

`RETR_EXTERNAL` (used when extracting polygons from RLE) only keeps outer contours — a mask with a genuine hole (a bird occluded through its middle) gets silently filled in once converted. Not fixable within YOLO-seg's own polygon format, which has no hole representation at all — a small, permanent gap specific to this dataset's occlusion pattern.

### 5. Before Phase 3 training

`data/coco.py` should reuse this notebook's disk-cached RLE→polygon pattern (`annotations_polygon_cache/`) — otherwise every `train`/`tune`/`sweep` run re-pays the ~1–2 min decode cost this notebook now only pays once.

### Next step

Prototype a first `yolo26n-seg` fit — separate notebook/step, once the GPU's free. See `plan.md` Phase 3.